Extraindo PDF

In [1]:
!pip install transformers datasets pdfplumber accelerate sentencepiece


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import pdfplumber
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:

def extract_text_from_pdf(pdf_path, skip_pages=6):
    """Extrai texto de um arquivo PDF."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            if i < skip_pages: # Ignora as primeiras páginas (capa, sumário, etc.)
                continue
            page_text = page.extract_text()
            if page_text:
                pages.append(page_text.strip())
    return "\n\n".join(pages)

# Substitua pelo caminho do seu PDF
pdf_path = "20260018701.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Total de caracteres extraídos: 243128

--- INÍCIO DO TEXTO ---

Amedicinadasimplicidade|7
A medicina da simplicidade
– trabalhar com plantas é a
ciência do simples
Alésio dos Passos Santos – ambientalista, cultivador e edu-
cadordeplantasmedicinais(comcontribuiçõesdeCésarPaulo
Simionatto,MuriloLeandroMarcos,LeilaNerydosSantosSouza,
Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon
Bittencourt)
Amedicinadosinergismoeaenergiadecura
Osinergismoéumadascoisasmaisimportantesnomundo
dasplantas.Oremédiodefarmácia,oquequeé?Umprincípio
ativoisolado,concentrad


Dividindo em chunks

In [38]:
import re

def limpar_texto(texto):
    """
    Remove espaços extras e normaliza o texto bruto extraído do PDF.
    Isso ajuda a eliminar ruídos comuns em extrações de arquivos PDF.
    """
    # Substitui múltiplas quebras de linha ou espaços por um único espaço e remove espaços no início e no fim
    return re.sub(r'\s+', ' ', texto).strip()

def dividir_em_paragrafos(texto):
    """
    Divide o texto em parágrafos significativos.
    """
    # Dividindo os paragrafos por quebra de linha e limpando espaços extras
    paragrafos = texto.split("\n")
    # Filtro de ruído: ignora fragmentos menores que 50 caracteres (como números de página) 
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]


In [39]:
def chunk_text_especializado(paragrafos, max_chunk_chars=500, overlap_chars=100):
    """
    Agrupa parágrafos em blocos menores (chunks) com sobreposição.
    
    Sugestões aplicadas:
    - max_chunk_chars=500: Chunks menores tendem a ser mais focados.
    - overlap_chars: Evita a perda de contexto nas 'emendas' entre blocos.
    """
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos:
        # TRATAMENTO DE INTEGRIDADE: 
        # Se um parágrafo sozinho for maior que o limite máximo, ele é dividido em partes.
        if len(paragrafo) > max_chunk_chars:
            if chunk_atual:
                chunks.append(chunk_atual)
                chunk_atual = ""
            # Divide o parágrafo gigante respeitando o overlap
            for i in range(0, len(paragrafo), max_chunk_chars - overlap_chars):
                chunks.append(paragrafo[i : i + max_chunk_chars])
            continue

        # Lógica de agrupamento com verificação de limite
        if len(chunk_atual) + len(paragrafo) + 1 <= max_chunk_chars:
            chunk_atual += (" " if chunk_atual else "") + paragrafo 
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            
            # APLICAÇÃO DE SOBREPOSIÇÃO (Sliding Window):
            # O novo chunk começa com o final do chunk anterior para manter o contexto [2].
            inicio_com_contexto = chunk_atual[-overlap_chars:] if len(chunk_atual) > overlap_chars else ""
            chunk_atual = (inicio_com_contexto + " " if inicio_com_contexto else "") + paragrafo

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


In [40]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 3085


In [41]:
chunks = chunk_text_especializado(paragrafos, max_chunk_chars=600)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:600]}...")

Número de chunks: 376

Exemplo de chunk (primeiro):
Alésio dos Passos Santos – ambientalista, cultivador e edu- cadordeplantasmedicinais(comcontribuiçõesdeCésarPaulo Simionatto,MuriloLeandroMarcos,LeilaNerydosSantosSouza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon ativoisolado,concentrado,sintetizado,quefazoremédio.Ea planta,oquequefaz?Todososprincípiosativosjuntosemsin- ergismo,trabalhandojuntos,éaenergiadaplantaquenãose vê em microscópio. Então pra mim, isolar o princípio ativo é coisasmaisimportantesnasplantas.Nuncavilivrocomesse Nãoadiantaestudarsomenteosgruposdeprincípiosativos....


Carregando modelo

In [42]:
# --- Modelo Causal ---
causal_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
causal_tokenizer = AutoTokenizer.from_pretrained(causal_id)
causal_tokenizer.pad_token = causal_tokenizer.eos_token  # GPT-2 não tem pad_token
causal_model = AutoModelForCausalLM.from_pretrained(causal_id)

generator_causal = pipeline(
    "text-generation",
    model=causal_model,
    tokenizer=causal_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3
)

print("Modelo carregado com sucesso.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2661.67it/s]


Modelo carregado com sucesso.


In [48]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1) #
    if strange_ratio > 0.3:
        return True
    return False

def run_model(generator, prompt):
    """Executa qualquer pipeline e retorna o texto gerado."""
    output = generator(prompt, max_new_tokens=256, temperature=0.3, do_sample=True)
    if isinstance(output, list):
        item = output[0]
        if isinstance(item, dict):
            return item.get('generated_text') or item.get('summary_text') or str(item)
        return str(item)
    return str(output)

def extract_triple(generated_text, chunk):
    """Tenta extrair a tripla da saída do modelo."""
    # 1. Tenta JSON
    json_match = re.search(r'\{[\s\S]*\}', generated_text)
    if json_match:
        try:
            data = json.loads(json_match.group(0))
            instruction = str(data.get("instruction", "")).strip()
            output = str(data.get("output", "")).strip()
            if instruction and output:
                return {"Instruction": instruction, "Output": output}
        except:
            pass
    # 2. Regex por campos
    instruction = output = ""
    for key in ["Instruction", "Output"]:
        pattern = rf'{key}\s*[:=-]\s*(.*)'
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            if key == "Instruction": instruction = match.group(1).strip()
            elif key == "Output": output = match.group(1).strip()

    if instruction and output:
        return {"Instruction": instruction, "Output": output} # Se ambos foram encontrados, retornamos imediatamente
    
    # 3. Se não encontrou nada útil, descarta
    return None
    


In [49]:
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")

Chunks válidos: 376


In [50]:
PROMPT_CAUSAL = """<|system|>
Você é um especialista em plantas medicinais. Leia o trecho e gere uma pergunta e resposta sobre ele.
</s>
<|user|>
Trecho:
{chunk}

Gere um JSON com:
- "instruction": pergunta objetiva sobre o trecho acima
- "output": resposta baseada apenas no trecho acima

Responda APENAS com o JSON.
</s>
<|assistant|>
{{"""

In [51]:
triplas_causal = []
amostra_chunks = clean_chunks[:30]  # para demonstração, processamos apenas 30 chunks

for i, chunk in enumerate(amostra_chunks):
    prompt = PROMPT_CAUSAL.replace("{chunk}", chunk)
    try:
        result = run_model(generator_causal, prompt)
        # O modelo causal pode repetir o prompt; removemos a parte inicial se existir
        if result.startswith(prompt):
            generated = result[len(prompt):].strip()
        else:
            generated = result
        triple = extract_triple(generated, chunk)
        if triple:  # só adiciona se extraiu algo real
            triplas_causal.append(triple)
            
        # if len(triple["output"]) > 20 and len(triple["instruction"]) > 5:
            # triplas_causal.append(triple)
            
        if i % 10 == 0:
            print(f"Processados {i}/{len(amostra_chunks)} chunks (causal)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (causal): {len(triplas_causal)}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processados 0/30 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 10/30 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 20/30 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Triplas geradas (causal): 1


In [52]:
validos = 0

for i, chunk in enumerate(amostra_chunks[:5]):
    prompt = PROMPT_CAUSAL.replace("{chunk}", chunk)

    result = run_model(generator_causal, prompt)

    if result.startswith(prompt):
        generated = result[len(prompt):].strip()
    else:
        generated = result

    print(f"\nCHUNK {i}")
    print("-"*50)
    print(generated[:1000])

    triple = extract_triple(generated, chunk)

    if triple:
        validos += 1

print("VALIDOS:", validos)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 0
--------------------------------------------------
"instruction": "Given the text:
Alésio dos Passos Santos – ambientalista, cultivador e edu- cadordeplantasmedicinais(comcontribuiçõesdeCésarPaulo Simionatto,MuriloLeandroMarcos,LeilaNerydosSantosSouza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon ativoisolado,concentrado,sintetizado,quefazoremédio.Ea planta,oquequefaz?
Todosprincípiosativosjuntosemsin- ergismo,trabalhandojuntos,éaenergiadaplantaquenãose vê em microscópio. Então pra mim, isolar o princípio ativo é coisasmaisimportantesnasplantas.Nuncavilivrocomesse Nãoadiantaestudarsomenteosgruposdeprincípiosativos.

Output:
[
  {
    "question": "What is the purpose of isolating the active principle in plants?",
    "


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 1
--------------------------------------------------
"instruction": "Pergunta objetiva sobre o trecho acima",
  "output": {
    "question": "Suma de todas as plantas medicinais",
    "answer": "Não existe um fitocomplexo, apenas um fitocom- plexo, como que as plantas se unem em um fitocomplexo."
  }
}}

O trecho acima é uma pergunta objetiva sobre o fitocomplexo de plantas medicinais. A resposta baseada apenas no trecho acima é: "Não existe um fitocomplexo, apenas um fitocom- plexo, como que as plantas se unem em um fitocomplexo."


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 2
--------------------------------------------------
"instruction": "Given the text: Pontes entre níveis de densidade dentro do campo de energia da planta. As pessoas tem de olhar pra planta, conhecer a planta, o comportamento dela, tem que conversarcomaplanta, como a saúde, a saúde mental, a saúde espiritual, a saúde física, a saúde do corpo, a saúde do espírito, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo, a saúde do mundo


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 3
--------------------------------------------------
"instruction": "Give a brief explanation of the topic discussed in the given text, focusing on the main points and key takeaways.",
  "output": {
    "response": {
      "question": "What is the importance of understanding the genetic makeup of a person in the context of healthcare?",
      "answer": "The importance of understanding the genetic makeup of a person in the context of healthcare is that it allows for the identification of predispositions and risk factors for various health conditions. By understanding the genetic makeup, healthcare professionals can tailor their treatments and preventive measures to the individual's unique needs and circumstances."
    }
  }
}}

CHUNK 4
--------------------------------------------------
"instruction": "Given the text: A sample example: The harvest of maca. On Holy Saturday, before Easter Sunday, the maca is gathered from the orchard. On the sixth day of Easter, the two maca plants

In [33]:
def clean_triple(triple):
    for key in triple:
        triple[key] = triple[key].strip()
    if triple["input"].lower() in ("none", "null", "n/a", ""):
        triple["input"] = ""
    if len(triple["output"]) < 20 or len(triple["instruction"]) < 5:
        return None
    return triple

def limpar_lista(triplas):
    return [t for t in (clean_triple(x) for x in triplas) if t is not None] # Filtra triplas inválidas após limpeza


triplas_causal = limpar_lista(triplas_causal)

print(f"Triplas causal após limpeza: {len(triplas_causal)}")

Triplas causal após limpeza: 44


In [34]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_causal_500chars.jsonl")

Dataset salvo em dataset_causal_500chars.jsonl


In [35]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "TinyLlama/TinyLlama-1.1B-Chat-v1.0")


=== TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===

--- Exemplo 1 ---
{
  "instruction": "pergunta que um usuário faria sobre o texto",
  "input": "",
  "output": "exemplo de resposta para a pergunta"
}

--- Exemplo 2 ---
{
  "instruction": "Pergunta que um usuário faria sobre o texto",
  "input": "",
  "output": "O texto contém informações sobre a cura das plantas, o que significa que a inter-relação pode ser comuns entre a área de ciências biológicas e a área de ciências da saúde."
}

--- Exemplo 3 ---
{
  "instruction": "Explique o conteúdo do texto.",
  "input": "eberBiavatti:ProfessoraDoutoradoDepartamento Mayara Krasinski Caddah: Professora Doutora do Departa- MelissaCostaSantos:FarmacêuticadaPrefeituraMunicipalde FlorianópolisedaComissãodePráticasIntegrativaseComple- MuriloLeandroMarcos:Médicodefamíliaecomunidadedocen- PráticasIntegrativaseComplementaresdeFlorianópolis. ShirleyRosa:FarmacêuticacolaboradoradoHortodePlantas Prainha - Cláudia Schenya, Fernanda Nowak, Gisele Damian Antonio